# CellScope: Breast Tumour Classification — BITS Virtual Lab Notebook

This notebook independently implements five classifiers for the Wisconsin Diagnostic Breast Cancer dataset.

**Target encoding:** `0 = malignant`, `1 = benign`.  
**Split:** stratified 80% training / 20% testing, `random_state=27`.  
Run all cells in order. The notebook saves trained model artifacts, the test CSV and a metrics table for the Streamlit app.


In [ ]:
# 1. Import packages
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, ConfusionMatrixDisplay,
    classification_report, RocCurveDisplay
)

print("Imports completed.")


In [ ]:
# 2. Load the supplied public dataset
data = pd.read_csv("dataset.csv")
print("Dataset dimensions:", data.shape)
display(data.head())
print("Missing values:", int(data.isna().sum().sum()))
print("Class counts (0=malignant, 1=benign):")
display(data["target"].value_counts().sort_index())


In [ ]:
# 3. Separate predictors and outcome; create a reproducible holdout set
X = data.drop(columns=["target"])
y = data["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=27, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Predictor count:", X.shape[1])


In [ ]:
# 4. Configure the five classifiers
estimators = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=3000, C=0.8, random_state=27))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DecisionTreeClassifier(max_depth=4, min_samples_leaf=4, random_state=27))
    ]),
    "K-Nearest Neighbors": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=7, weights="distance"))
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", GaussianNB(var_smoothing=1e-8))
    ]),
    "Support Vector Machine": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("classifier", SVC(C=1.5, kernel="rbf", probability=True, random_state=27))
    ])
}
print("Configured classifiers:", list(estimators))


In [ ]:
# 5. Train, evaluate and export each classifier
artifact_names = {
    "Logistic Regression": "logistic_regression",
    "Decision Tree": "decision_tree",
    "K-Nearest Neighbors": "knn",
    "Gaussian Naive Bayes": "gaussian_nb",
    "Support Vector Machine": "svm"
}
Path("model").mkdir(exist_ok=True)
fitted_models = {}
metric_rows = []

for label, estimator in estimators.items():
    estimator.fit(X_train, y_train)
    fitted_models[label] = estimator
    predicted = estimator.predict(X_test)
    probability = estimator.predict_proba(X_test)[:, 1]
    metric_rows.append({
        "Model": label,
        "Accuracy": accuracy_score(y_test, predicted),
        "AUC": roc_auc_score(y_test, probability),
        "Precision": precision_score(y_test, predicted, zero_division=0),
        "Recall": recall_score(y_test, predicted, zero_division=0),
        "F1": f1_score(y_test, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_test, predicted)
    })
    joblib.dump(estimator, Path("model") / f"{artifact_names[label]}.joblib")

metrics_table = pd.DataFrame(metric_rows)
display(metrics_table.round(4))
metrics_table.to_csv("model_metrics.csv", index=False)
print("Five model files and model_metrics.csv saved.")


In [ ]:
# 6. Visualize the metric comparison
metric_columns = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
ax = metrics_table.set_index("Model")[metric_columns].plot(
    kind="bar", figsize=(12, 6), ylim=(0, 1.05), rot=20
)
ax.set_title("Holdout performance across five classifiers")
ax.set_ylabel("Score")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 7. Confusion matrices and classification reports
for label, estimator in fitted_models.items():
    predicted = estimator.predict(X_test)
    print("\\n" + "=" * 62)
    print(label)
    print(classification_report(
        y_test, predicted,
        target_names=["Malignant", "Benign"],
        zero_division=0
    ))
    ConfusionMatrixDisplay.from_predictions(
        y_test, predicted,
        display_labels=["Malignant", "Benign"],
        cmap="Purples"
    )
    plt.title(f"Confusion matrix — {label}")
    plt.tight_layout()
    plt.show()


In [ ]:
# 8. ROC curves for all models
fig, ax = plt.subplots(figsize=(8, 6))
for label, estimator in fitted_models.items():
    RocCurveDisplay.from_estimator(estimator, X_test, y_test, name=label, ax=ax)
ax.set_title("ROC curve comparison")
plt.tight_layout()
plt.show()


In [ ]:
# 9. Export the held-out test data for Streamlit
test_export = X_test.copy()
test_export["target"] = y_test.values
test_export.to_csv("test_data.csv", index=False)

print("Exported test_data.csv with", len(test_export), "rows.")
print("Exported files:", sorted(p.name for p in Path("model").glob("*.joblib")))


## Interpretation prompts

Write your own observations after examining the metric table and plots:

- Which model has the highest AUC?
- Which model has the highest recall for the positive class (benign)?
- Does the Decision Tree perform differently from the scaled models?
- Which model gives the most balanced performance according to F1 and MCC?
- Explain why the best model on this particular test split should not automatically be treated as best for clinical use.

Do not treat these models as medical diagnostic tools.
